**Import the Raw Data and Define File Paths for four primary data**

In [57]:
##import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [58]:
project_dir = Path.cwd().parent

##data directories
raw_dir = project_dir / "data" / "Raw"
processed_dir = project_dir / "data" / "Processed"

##file paths
solar_file = raw_dir / "mart_solar_exports_full_release_monthly.csv"
cleantech_file = raw_dir / "712clean_tech_exports_full_release_monthly.xlsx"
irena_file = raw_dir / "IRENA.xlsx"
population_file = raw_dir / "712population_712API_SP.POP.TOTL_DS2_en_csv_v2_5254.xlsx"

print("Solar exports:", solar_file.exists())
print("Cleantech exports:", cleantech_file.exists())
print("IRENA:", irena_file.exists())
print("Population:", population_file.exists())

Solar exports: True
Cleantech exports: True
IRENA: True
Population: True


In [59]:
solar_raw = pd.read_csv(solar_file)
##cleantech
cleantech_solar_raw = pd.read_excel(
    cleantech_file,
    sheet_name="Solar PV"
)

cleantech_all_raw = pd.read_excel(
    cleantech_file,
    sheet_name="All technologies"
)

##Electricity Statistics
irena_raw = pd.read_excel(
    irena_file,
    sheet_name="ELEC-C",
    skiprows=2,
    header=None,
    names=[
        "Country/area",
        "Technology",
        "Data Type",
        "Grid connection",
        "Year",
        "Value"
    ]
)

##Population
population_raw = pd.read_excel(population_file)


In [60]:
print("Solar Exports:", solar_raw.shape)
print("Cleantech Solar PV:", cleantech_solar_raw.shape)
print("Cleantech All Technologies:", cleantech_all_raw.shape)
print("IRENA:", irena_raw.shape)
print("Population:", population_raw.shape)

Solar Exports: (31254, 13)
Cleantech Solar PV: (4764, 7)
Cleantech All Technologies: (4764, 7)
IRENA: (21530, 6)
Population: (266, 12)


**Preliminary Data Review and Quality Audit**

In [61]:
##Inspect Column Names and Data Types
datasets = {
    "Solar Exports": solar_raw,
    "Cleantech Solar PV": cleantech_solar_raw,
    "Cleantech All Technologies": cleantech_all_raw,
    "IRENA": irena_raw,
    "Population": population_raw
}


solar_raw.info()
cleantech_solar_raw.info()
cleantech_all_raw.info()
irena_raw.info()
population_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 31254 entries, 0 to 31253
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Area                      31254 non-null  str    
 1   Date                      31254 non-null  str    
 2   Area type                 31254 non-null  str    
 3   Region                    28391 non-null  str    
 4   Commodity code            31254 non-null  int64  
 5   Commodity type            31254 non-null  str    
 6   Amount (items)            31254 non-null  int64  
 7   Amount (kg)               31254 non-null  float64
 8   Amount (USD)              31254 non-null  float64
 9   Commodity price           31254 non-null  float64
 10  Price unit                31254 non-null  str    
 11  Capacity (MW)             31254 non-null  float64
 12  Cumulative capacity (MW)  31254 non-null  float64
dtypes: float64(5), int64(2), str(6)
memory usage: 3.1 MB
<class 'pandas.Data

In [62]:
##Check Missing Values
##Solar Exports
print(solar_raw.isna().sum())

##Cleantech Solar PV
print(cleantech_solar_raw.isna().sum())

##Cleantech All Technologies
print(cleantech_all_raw.isna().sum())

##IRENA
print(irena_raw.isna().sum())

##Population
print(population_raw.isna().sum())


##Check Duplicate Rows

##Solar Exports
print("Solar Exports:", solar_raw.duplicated().sum())

##Cleantech Solar PV
print("Cleantech Solar PV:", cleantech_solar_raw.duplicated().sum())

##Cleantech All Technologies
print("Cleantech All Technologies:", cleantech_all_raw.duplicated().sum())

##IRENA
print("IRENA:", irena_raw.duplicated().sum())

##Population
print("Population:", population_raw.duplicated().sum())


Area                           0
Date                           0
Area type                      0
Region                      2863
Commodity code                 0
Commodity type                 0
Amount (items)                 0
Amount (kg)                    0
Amount (USD)                   0
Commodity price                0
Price unit                     0
Capacity (MW)                  0
Cumulative capacity (MW)       0
dtype: int64
Area                                   0
Date                                   0
Area type                              0
Region                                 0
Commodity category                     0
Amount (USD)                           0
Amount 12 month rolling sum (USD)    628
dtype: int64
Area                                   0
Date                                   0
Area type                              0
Region                                 0
Commodity category                     0
Amount (USD)                           0
Amount 12 mo

In [63]:
##Inspect IRENA Data
print(irena_raw.head())
print(irena_raw.tail())

print((irena_raw["Value"] == "..").sum())

  Country/area          Technology                     Data Type  \
0  Afghanistan  Solar photovoltaic  Electricity Generation (GWh)   
1          NaN                 NaN                           NaN   
2          NaN                 NaN                           NaN   
3          NaN                 NaN                           NaN   
4          NaN                 NaN                           NaN   

  Grid connection    Year  Value  
0             All  2018.0  38.09  
1             NaN  2019.0  57.12  
2             NaN  2020.0  64.55  
3             NaN  2021.0  76.36  
4             NaN  2022.0  89.84  
                                     Country/area Technology Data Type  \
21525                                   Database:        NaN       NaN   
21526  IRENA Renewable Energy Statistics database        NaN       NaN   
21527                                         NaN        NaN       NaN   
21528                    Internal reference code:        NaN       NaN   
21529      

It used pandas to examine the four main datasets, looking at their dimensions, column names, data types, missing values, and duplicate records.

The IRENA dataset was processed separately because its original Excel format contained missing category labels, special markers for missing values, and additional comment rows.

The purpose of these preliminary checks was to identify potential data quality issues before filtering, cleaning, and merging the datasets.

The initial data quality audit identified several issues across the four primary datasets.

The Ember Solar Exports dataset contains 2,863 missing values in the Region column, while the other variables have no missing values. Both Ember Cleantech datasets contain 628 missing values in the 12-month rolling sum column, but their monthly export values are complete.

The IRENA dataset requires additional cleaning because its original Excel format contains blank category labels, special missing-value markers (".."), and 26 non-data rows. The initial duplicate check identified 14,855 duplicate rows before restoring the grouped category labels. Duplicate observations will be reassessed after cleaning.

The World Bank Population dataset contains one missing value for each year from 2018 to 2025. Missing values within the final study area will be checked after filtering.

These issues will be addressed or documented during the data cleaning process.

**Define Study Area and Filter Data**

In [64]:
## Define Study Countries
study_countries = [
    "United States",
    "Canada",
    "Mexico",
    "Argentina",
    "Bolivia",
    "Brazil",
    "Chile",
    "Colombia",
    "Ecuador",
    "Guyana",
    "Paraguay",
    "Peru",
    "Suriname",
    "Uruguay",
    "Venezuela"
]

print("Number of study countries:", len(study_countries))

Number of study countries: 15


We hope to select observations for the 15 study countries from 2018 to 2025.

In [65]:
##Filter Ember Solar Exports

solar_clean = solar_raw.copy()

##Convert Date to datetime
solar_clean["Date"] = pd.to_datetime(solar_clean["Date"])

solar_clean["Year"] = solar_clean["Date"].dt.year

##Filter study countries and years
solar_clean = solar_clean.loc[
    (solar_clean["Area"].isin(study_countries)) &
    (solar_clean["Year"] >= 2018) &
    (solar_clean["Year"] <= 2025)
].copy()

print("Shape:", solar_clean.shape)
print("\nCountries:")
print(solar_clean["Area"].unique())
print("\nYears:")
print(solar_clean["Year"].unique())

Shape: (1922, 14)

Countries:
<StringArray>
[    'Argentina',       'Bolivia',        'Brazil',        'Canada',
         'Chile',      'Colombia',       'Ecuador',        'Guyana',
        'Mexico',      'Paraguay',          'Peru',      'Suriname',
 'United States',       'Uruguay',     'Venezuela']
Length: 15, dtype: str

Years:
[2018 2019 2020 2021 2022 2023 2024 2025]


In [66]:
##Inspect Filtered Solar Exports
print("Number of countries:", solar_clean["Area"].nunique())
print("\nCommodity Types:")
print(solar_clean["Commodity type"].value_counts())

print("\nMissing Values:")
print(solar_clean.isna().sum())

solar_clean.head()

Number of countries: 15

Commodity Types:
Commodity type
Panels    1375
Cells      475
Wafers      72
Name: count, dtype: int64

Missing Values:
Area                        0
Date                        0
Area type                   0
Region                      0
Commodity code              0
Commodity type              0
Amount (items)              0
Amount (kg)                 0
Amount (USD)                0
Commodity price             0
Price unit                  0
Capacity (MW)               0
Cumulative capacity (MW)    0
Year                        0
dtype: int64


,Area,Date,Area type,Region,Commodity code,Commodity type,Amount (items),Amount (kg),Amount (USD),Commodity price,Price unit,Capacity (MW),Cumulative capacity (MW),Year
674,Argentina,2018-01-01,Country or economy,Latin America and Caribbean,85414020,Panels,86123,2047695.0,8419251.0,0.29,USD/W,29.11,141.30,2018
675,Argentina,2018-02-01,Country or economy,Latin America and Caribbean,85414020,Panels,113355,2651520.0,10686024.0,0.28,USD/W,37.75,179.05,2018
676,Argentina,2018-03-01,Country or economy,Latin America and Caribbean,85414020,Panels,42825,1110636.0,4819506.0,0.28,USD/W,17.40,196.45,2018
677,Argentina,2018-04-01,Country or economy,Latin America and Caribbean,85414020,Panels,78464,2036339.0,8789341.0,0.27,USD/W,32.46,228.92,2018
678,Argentina,2018-05-01,Country or economy,Latin America and Caribbean,85414020,Panels,54898,1306926.0,5729775.0,0.26,USD/W,21.65,250.57,2018


**Filter Solar Panels**

We want to select solar PV panel exports for the main analysis.

In [67]:
##Filter Solar Panels

solar_panels = solar_clean.loc[
    solar_clean["Commodity type"] == "Panels"
].copy()

print(solar_panels.shape)

solar_panels.head()

##Aggregate Monthly Exports to Annual Exports

solar_annual = solar_panels.groupby(
    ["Area", "Year"], as_index=False
)[["Amount (USD)", "Capacity (MW)"]].sum()

print(solar_annual.shape)

solar_annual.head()

(1375, 14)
(120, 4)


,Area,Year,Amount (USD),Capacity (MW)
0,Argentina,2018,171072125.0,665.24
1,Argentina,2019,134984757.0,496.16
2,Argentina,2020,12443409.0,52.89
3,Argentina,2021,33380468.0,126.72
4,Argentina,2022,40100211.0,149.13


In [68]:
##Rename Variables

solar_annual = solar_annual.rename(columns={
    "Amount (USD)": "China_panel_exports_USD",
    "Capacity (MW)": "China_panel_exports_MW"
})

solar_annual.head()

,Area,Year,China_panel_exports_USD,China_panel_exports_MW
0,Argentina,2018,171072125.0,665.24
1,Argentina,2019,134984757.0,496.16
2,Argentina,2020,12443409.0,52.89
3,Argentina,2021,33380468.0,126.72
4,Argentina,2022,40100211.0,149.13


**Data Quality Check**

In [69]:
##Check Annual Solar Export Data
print(solar_annual.shape)

##Check missing values
print(solar_annual.isna().sum())

##Check duplicate Country-Year records
print(solar_annual.duplicated(
    subset=["Area", "Year"]
).sum())

print(solar_annual["Area"].nunique())
print(solar_annual["Year"].nunique())


(120, 4)
Area                       0
Year                       0
China_panel_exports_USD    0
China_panel_exports_MW     0
dtype: int64
0
15
8


In [70]:
##Check Monthly Coverage

##Extract month
solar_panels["Month"] = solar_panels["Date"].dt.month

##Count months for each country and year
solar_months = solar_panels.groupby(
    ["Area", "Year"]
)["Month"].nunique().reset_index()

##Rename column
solar_months = solar_months.rename(columns={
    "Month": "Months_reported"
})

##Check countries and years with fewer than 12 months
solar_months.loc[
    solar_months["Months_reported"] < 12
]


,Area,Year,Months_reported
8,Bolivia,2018,8
9,Bolivia,2019,7
10,Bolivia,2020,10
11,Bolivia,2021,10
12,Bolivia,2022,9
13,Bolivia,2023,11
56,Guyana,2018,11
57,Guyana,2019,11
60,Guyana,2022,11
61,Guyana,2023,11


**Clean Ember Cleantech Exports**

In [71]:
##Inspect Cleantech Datasets

print(cleantech_solar_raw.head())
print(cleantech_all_raw.head())

##Check commodity categories
print(cleantech_solar_raw["Commodity category"].unique())

print(cleantech_all_raw["Commodity category"].unique())

       Area       Date           Area type                       Region  \
0  Anguilla 2023-01-01  Country or economy  Latin America and Caribbean   
1  Anguilla 2023-02-01  Country or economy  Latin America and Caribbean   
2  Anguilla 2023-03-01  Country or economy  Latin America and Caribbean   
3  Anguilla 2023-05-01  Country or economy  Latin America and Caribbean   
4  Anguilla 2023-06-01  Country or economy  Latin America and Caribbean   

  Commodity category  Amount (USD)  Amount 12 month rolling sum (USD)  
0           Solar PV             0                                NaN  
1           Solar PV             0                                NaN  
2           Solar PV             0                                NaN  
3           Solar PV          6212                                NaN  
4           Solar PV          2872                                NaN  
       Area       Date           Area type                       Region  \
0  Anguilla 2023-01-01  Country or economy

**Update Date Types and Filter Data**

In [72]:
##Update Data Types
cleantech_solar = cleantech_solar_raw.copy()
cleantech_all = cleantech_all_raw.copy()

# Convert Date to datetime
cleantech_solar["Date"] = pd.to_datetime(cleantech_solar["Date"])

cleantech_all["Date"] = pd.to_datetime(cleantech_all["Date"])

# Extract Year
cleantech_solar["Year"] = cleantech_solar["Date"].dt.year

cleantech_all["Year"] = cleantech_all["Date"].dt.year

In [73]:
##Filter Study Countries and Years

##Filter Cleantech Solar PV
cleantech_solar = cleantech_solar.loc[
    (cleantech_solar["Area"].isin(study_countries)) &
    (cleantech_solar["Year"] >= 2018) &
    (cleantech_solar["Year"] <= 2025)
].copy()

##Filter Cleantech All Technologies
cleantech_all = cleantech_all.loc[
    (cleantech_all["Area"].isin(study_countries)) &
    (cleantech_all["Year"] >= 2018) &
    (cleantech_all["Year"] <= 2025)
].copy()

print("Cleantech Solar PV:", cleantech_solar.shape)

print("Cleantech All Technologies:", cleantech_all.shape)


Cleantech Solar PV: (1439, 8)
Cleantech All Technologies: (1439, 8)


**Aggregate monthly data into annual data**

In [74]:
##Aggregate Solar PV Exports

cleantech_solar_annual = cleantech_solar.groupby(
    ["Area", "Year"], as_index=False
)[["Amount (USD)"]].sum()

##Rename variable
cleantech_solar_annual = cleantech_solar_annual.rename(
    columns={
        "Amount (USD)": "CleanTech_solarPV_USD"
    }
)

print(cleantech_solar_annual.shape)

cleantech_solar_annual.head()

(120, 3)


,Area,Year,CleanTech_solarPV_USD
0,Argentina,2018,171072125
1,Argentina,2019,134984757
2,Argentina,2020,12443409
3,Argentina,2021,33380468
4,Argentina,2022,40233974


In [75]:
##Aggregate All Technology Exports

cleantech_all_annual = cleantech_all.groupby(
    ["Area", "Year"], as_index=False
)[["Amount (USD)"]].sum()

##Rename variable
cleantech_all_annual = cleantech_all_annual.rename(
    columns={
        "Amount (USD)": "CleanTech_alltech_USD"
    }
)

print(cleantech_all_annual.shape)

cleantech_all_annual.head()


(120, 3)


,Area,Year,CleanTech_alltech_USD
0,Argentina,2018,636153927
1,Argentina,2019,586387276
2,Argentina,2020,255363464
3,Argentina,2021,333211051
4,Argentina,2022,417618123


We want to merge the two cleantech datasets to combine annual Solar PV and All Technology export values.

In [76]:
##Merge Cleantech Datasets

cleantech_annual = pd.merge(
    cleantech_solar_annual,
    cleantech_all_annual,
    on=["Area", "Year"],
    how="inner",
    validate="one_to_one"
)

print(cleantech_annual.shape)

cleantech_annual.head()


(120, 4)


,Area,Year,CleanTech_solarPV_USD,CleanTech_alltech_USD
0,Argentina,2018,171072125,636153927
1,Argentina,2019,134984757,586387276
2,Argentina,2020,12443409,255363464
3,Argentina,2021,33380468,333211051
4,Argentina,2022,40233974,417618123


In [77]:
##Data Quality Check

print(cleantech_annual.shape)

##Check missing values
print(cleantech_annual.isna().sum())

##Check duplicated country-year records
print(cleantech_annual.duplicated(
    subset=["Area", "Year"]
).sum())

##Check number of countries and years
print(cleantech_annual["Area"].nunique())
print(cleantech_annual["Year"].nunique())


(120, 4)
Area                     0
Year                     0
CleanTech_solarPV_USD    0
CleanTech_alltech_USD    0
dtype: int64
0
15
8


In [78]:
##Check Monthly Coverage

##Extract month
cleantech_solar["Month"] = cleantech_solar["Date"].dt.month

##Count months for each country and year
cleantech_months = cleantech_solar.groupby(
    ["Area", "Year"]
)["Month"].nunique().reset_index()

cleantech_months = cleantech_months.rename(
    columns={"Month": "Months_reported"}
)

##Find incomplete years
cleantech_months.loc[
    cleantech_months["Months_reported"] < 12
]


,Area,Year,Months_reported
90,Suriname,2020,11


In [79]:
##Check Merge Results

cleantech_check = pd.merge(
    cleantech_solar_annual,
    cleantech_all_annual,
    on=["Area", "Year"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

##Check matched and unmatched records
print(cleantech_check["_merge"].value_counts())

_merge
both          120
left_only       0
right_only      0
Name: count, dtype: int64


### Ember Cleantech Export Data Cleaning

The Ember China Cleantech Export dataset contains two worksheets: "Solar PV" and "All technologies." We filtered both datasets to retain observations from 2018 to 2025 for the 15 study countries in North and South America.

We converted the "Date" column to datetime format and extracted the calendar year to facilitate annual aggregation. Using the `groupby()` and `sum()` functions, we aggregated monthly export values (USD) by country and year. To avoid double-counting, we excluded the "Amount 12 month rolling sum (USD)" variable from the aggregation.

We merged the two annual datasets using country and year as join keys. The merged dataset contains 120 country-year observations and two annual export value variables: `CleanTech_solarPV_USD` and `CleanTech_alltech_USD`.

**Data Quality Assessment:** No missing values or duplicate country-year observations were identified in the merged annual dataset. However, the monthly data coverage check revealed one incomplete country-year combination: Suriname had export records for only 11 months in 2020. This observation was retained, and the missing month's value was not imputed.

The merge validation confirmed that all 120 country-year keys were successfully matched between the two datasets, with no unmatched records.

**Clean IRENA Solar Energy Data**

In [80]:
##Remove Non-Data Rows

irena_clean = irena_raw.copy()

##Convert Year to numeric
irena_clean["Year"] = pd.to_numeric(
    irena_clean["Year"], errors="coerce"
)

##Keep rows with valid years
irena_clean = irena_clean.loc[
    irena_clean["Year"].notna()
].copy()

##Convert Year to integer
irena_clean["Year"] = irena_clean["Year"].astype(int)

print(irena_clean.shape)

irena_clean.tail()

(21504, 6)


,Country/area,Technology,Data Type,Grid connection,Year,Value
21499,NaN,NaN,NaN,NaN,2021,..
21500,NaN,NaN,NaN,NaN,2022,..
21501,NaN,NaN,NaN,NaN,2023,..
21502,NaN,NaN,NaN,NaN,2024,..
21503,NaN,NaN,NaN,NaN,2025,..


In [81]:
##Fill Missing Category Labels

irena_clean["Country/area"] = irena_clean["Country/area"].ffill()

irena_clean["Technology"] = irena_clean["Technology"].ffill()

irena_clean["Data Type"] = irena_clean["Data Type"].ffill()

irena_clean["Grid connection"] = irena_clean["Grid connection"].ffill()

irena_clean.head(10)


,Country/area,Technology,Data Type,Grid connection,Year,Value
0,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2018,38.09
1,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2019,57.12
2,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2020,64.55
3,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2021,76.36
4,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2022,89.84
5,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2023,116.73
6,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2024,121.21
7,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),All,2025,..
8,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),On-grid,2018,1.11
9,Afghanistan,Solar photovoltaic,Electricity Generation (GWh),On-grid,2019,20.77


In [82]:
##Convert Values to Numeric

irena_clean["Value"] = pd.to_numeric(
    irena_clean["Value"], errors="coerce"
)

irena_clean.info()

##Check missing values
print(irena_clean.isna().sum())

##Check duplicate records using the original observation keys
print(
    irena_clean.duplicated(
        subset=[
            "Country/area",
            "Technology",
            "Data Type",
            "Grid connection",
            "Year"
        ]
    ).sum()
)


<class 'pandas.DataFrame'>
RangeIndex: 21504 entries, 0 to 21503
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Country/area     21504 non-null  str    
 1   Technology       21504 non-null  str    
 2   Data Type        21504 non-null  str    
 3   Grid connection  21504 non-null  str    
 4   Year             21504 non-null  int64  
 5   Value            8921 non-null   float64
dtypes: float64(1), int64(1), str(4)
memory usage: 1008.1 KB
Country/area           0
Technology             0
Data Type              0
Grid connection        0
Year                   0
Value              12583
dtype: int64
0


In [83]:
##Standardize Country Names

irena_clean["Country/area"] = irena_clean["Country/area"].replace({
    "USA": "United States"
})

print(
    irena_clean.loc[
        irena_clean["Country/area"] == "United States",
        "Country/area"
    ].unique()
)


<StringArray>
['United States']
Length: 1, dtype: str


In [84]:
##Filter Study Area and Solar PV Data

irena_pv = irena_clean.loc[
    (irena_clean["Country/area"].isin(study_countries)) &
    (irena_clean["Year"] >= 2018) &
    (irena_clean["Year"] <= 2025) &
    (irena_clean["Technology"] == "Solar photovoltaic") &
    (irena_clean["Grid connection"] == "All") &
    (irena_clean["Data Type"].isin([
        "Electricity Generation (GWh)",
        "Electrical Installed Capacity (MW)"
    ]))
].copy()

print(irena_pv.shape)

print(irena_pv["Data Type"].value_counts())
print(irena_pv["Country/area"].nunique())

irena_pv.head()


(240, 6)
Data Type
Electricity Generation (GWh)          120
Electrical Installed Capacity (MW)    120
Name: count, dtype: int64
15


,Country/area,Technology,Data Type,Grid connection,Year,Value
768,Argentina,Solar photovoltaic,Electricity Generation (GWh),All,2018,123.48
769,Argentina,Solar photovoltaic,Electricity Generation (GWh),All,2019,860.07
770,Argentina,Solar photovoltaic,Electricity Generation (GWh),All,2020,1464.14
771,Argentina,Solar photovoltaic,Electricity Generation (GWh),All,2021,2402.05
772,Argentina,Solar photovoltaic,Electricity Generation (GWh),All,2022,3137.39


In [85]:
##Select Installed Capacity Data

irena_capacity = irena_pv.loc[
    irena_pv["Data Type"] == "Electrical Installed Capacity (MW)",
    ["Country/area", "Year", "Value"]
].copy()

##Rename columns
irena_capacity = irena_capacity.rename(columns={
    "Country/area": "Area",
    "Value": "PV_installed_capacity_MW"
})

print(irena_capacity.shape)

irena_capacity.head()


(120, 3)


,Area,Year,PV_installed_capacity_MW
792,Argentina,2018,196.11
793,Argentina,2019,445.85
794,Argentina,2020,767.87
795,Argentina,2021,1076.92
796,Argentina,2022,1112.67


In [86]:
##Select Electricity Generation Data

irena_generation = irena_pv.loc[
    irena_pv["Data Type"] == "Electricity Generation (GWh)",
    ["Country/area", "Year", "Value"]
].copy()

##Rename columns
irena_generation = irena_generation.rename(columns={
    "Country/area": "Area",
    "Value": "PV_generation_GWh"
})

print(irena_generation.shape)

irena_generation.head()


(120, 3)


,Area,Year,PV_generation_GWh
768,Argentina,2018,123.48
769,Argentina,2019,860.07
770,Argentina,2020,1464.14
771,Argentina,2021,2402.05
772,Argentina,2022,3137.39


In [87]:
##Merge Capacity and Generation
irena_annual = pd.merge(
    irena_capacity,
    irena_generation,
    on=["Area", "Year"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print(irena_annual["_merge"].value_counts())

##Remove merge indicator
irena_annual = irena_annual.drop(columns=["_merge"])

print(irena_annual.shape)

irena_annual.head()


_merge
both          120
left_only       0
right_only      0
Name: count, dtype: int64
(120, 4)


,Area,Year,PV_installed_capacity_MW,PV_generation_GWh
0,Argentina,2018,196.11,123.48
1,Argentina,2019,445.85,860.07
2,Argentina,2020,767.87,1464.14
3,Argentina,2021,1076.92,2402.05
4,Argentina,2022,1112.67,3137.39


In [88]:
##Data Quality Audit
print(irena_annual.shape)

print(irena_annual.isna().sum())

##Check duplicated country-year records
print(
    irena_annual.duplicated(
        subset=["Area", "Year"]
    ).sum()
)

print("Countries:", irena_annual["Area"].nunique())
print("Years:", irena_annual["Year"].nunique())


(120, 4)
Area                         0
Year                         0
PV_installed_capacity_MW     0
PV_generation_GWh           15
dtype: int64
0
Countries: 15
Years: 8


In [89]:
##Check Implausible Values

##Check negative installed capacity
print(
    irena_annual.loc[
        irena_annual["PV_installed_capacity_MW"] < 0
    ]
)

##Check negative electricity generation
print(
    irena_annual.loc[
        irena_annual["PV_generation_GWh"] < 0
    ]
)

Empty DataFrame
Columns: [Area, Year, PV_installed_capacity_MW, PV_generation_GWh]
Index: []
Empty DataFrame
Columns: [Area, Year, PV_installed_capacity_MW, PV_generation_GWh]
Index: []


### IRENA Solar Energy Data Cleaning

We cleaned the IRENA (International Renewable Energy Agency) electricity statistics dataset and extracted data on solar PV installed capacity and electricity generation for 15 study countries from 2018 to 2025.

First, we removed 26 non-data rows containing source information and explanatory notes. For missing category labels, we used the `ffill()` function to impute values. All values were converted to numeric format, and missing observations were represented as `NaN`.

We standardized country names by uniformly replacing “USA” with “United States.” Next, we filtered the data to include only solar photovoltaic (PV) information, specifically electricity generation (GWh) and installed capacity (MW). In the “Grid Connection Type” category, we retained only the “All” entry to avoid double-counting both grid-connected and off-grid capacity.


The two data types were separated into individual DataFrames and merged using country and year as join keys. The resulting dataset contains 120 country-year observations with two variables: `PV_installed_capacity_MW` and `PV_generation_GWh`.

**Data Quality Assessment:** The final IRENA dataset contains no duplicated country-year observations or missing installed capacity values. However, electricity generation values are unavailable for all 15 countries in 2025. These missing values were retained as `NaN` rather than replaced with zero.

The merge validation confirmed that all 120 country-year keys matched between the installed capacity and electricity generation datasets, with no unmatched records.

**Clean World Bank Population Data**

In [90]:
##Inspect Population Data
print(population_raw.shape)
print(population_raw.columns)
population_raw.head()


(266, 12)
Index([  'Country Name',   'Country Code', 'Indicator Name', 'Indicator Code',
                   2018,             2019,             2020,             2021,
                   2022,             2023,             2024,             2025],
      dtype='object')


,Country Name,Country Code,Indicator Name,Indicator Code,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"Population, total",SP.POP.TOTL,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,108785.0
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,788844284.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,43844111.0
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,451395343.0,462522286.0,473687685.0,484978794.0,496366058.0,508318102.0,520655398.0,532809933.0
4,Angola,AGO,"Population, total",SP.POP.TOTL,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,39040039.0


In [91]:
##Standardize Country Names
population_clean = population_raw.copy()

##Standardize country names
population_clean["Country Name"] = population_clean["Country Name"].replace({
    "Venezuela, RB": "Venezuela"
})

print(population_clean["Country Name"].unique())


<StringArray>
[                      'Aruba', 'Africa Eastern and Southern',
                 'Afghanistan',  'Africa Western and Central',
                      'Angola',                     'Albania',
                     'Andorra',                  'Arab World',
        'United Arab Emirates',                   'Argentina',
 ...
       'Virgin Islands (U.S.)',                    'Viet Nam',
                     'Vanuatu',                       'World',
                       'Samoa',                      'Kosovo',
                 'Yemen, Rep.',                'South Africa',
                      'Zambia',                    'Zimbabwe']
Length: 266, dtype: str


In [92]:
##Filter Study Countries
population_clean = population_clean.loc[
    population_clean["Country Name"].isin(study_countries)
].copy()

print(population_clean.shape)

print(population_clean["Country Name"].unique())

print(population_clean.isna().sum())


(15, 12)
<StringArray>
[    'Argentina',       'Bolivia',        'Brazil',        'Canada',
         'Chile',      'Colombia',       'Ecuador',        'Guyana',
        'Mexico',          'Peru',      'Paraguay',      'Suriname',
       'Uruguay', 'United States',     'Venezuela']
Length: 15, dtype: str
Country Name      0
Country Code      0
Indicator Name    0
Indicator Code    0
2018              0
2019              0
2020              0
2021              0
2022              0
2023              0
2024              0
2025              0
dtype: int64


In [93]:
##Reshape Population Data
population_long = population_clean.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=range(2018, 2026),
    var_name="Year",
    value_name="Population"
)

print(population_long.shape)

population_long.head()


(120, 4)


,Country Name,Country Code,Year,Population
0,Argentina,ARG,2018,44654882.0
1,Bolivia,BOL,2018,11508368.0
2,Brazil,BRA,2018,206107261.0
3,Canada,CAN,2018,37072620.0
4,Chile,CHL,2018,18893191.0


In [94]:
##Select and Rename Variables

##Select necessary columns
population_annual = population_long.loc[
    :, ["Country Name", "Year", "Population"]
].copy()

##Rename country column
population_annual = population_annual.rename(columns={
    "Country Name": "Area"
})

##Convert year to integer
population_annual["Year"] = population_annual["Year"].astype(int)

##Sort by country and year
population_annual = population_annual.sort_values(
    ["Area", "Year"]
)

print(population_annual.shape)

population_annual.head()


(120, 3)


,Area,Year,Population
0,Argentina,2018,44654882.0
15,Argentina,2019,44973465.0
30,Argentina,2020,45191965.0
45,Argentina,2021,45312281.0
60,Argentina,2022,45407904.0


In [95]:
##Data Quality Audit

print(population_annual.shape)

print(population_annual.isna().sum())

##Check duplicate country-year observations
print(population_annual.duplicated(
    subset=["Area", "Year"]
).sum())

print("Countries:", population_annual["Area"].nunique())
print("Years:", population_annual["Year"].nunique())


(120, 3)
Area          0
Year          0
Population    0
dtype: int64
0
Countries: 15
Years: 8


In [96]:
##Check Implausible Values
population_annual.loc[
    population_annual["Population"] <= 0
]

,Area,Year,Population


In [97]:
population_annual["Population"].describe()

count    1.200000e+02
mean     6.205965e+07
std      9.114072e+07
min      5.995130e+05
25%      6.907901e+06
50%      2.842481e+07
75%      5.008849e+07
max      3.417849e+08
Name: Population, dtype: float64

### World Bank Population Data Cleaning

We cleaned and restructured the World Bank’s population dataset to obtain annual population data for the 15 study countries from 2018 to 2025.

First, we standardized country names by replacing “Venezuela, RB” with “Venezuela” to ensure consistency across datasets. Next, we filtered the dataset to retain only the 15 study countries located in the Americas.

The original population dataset was in “wide format,” meaning each year’s data occupied a separate column. We used the `melt()` function to convert it to “long format,” so that each country had one observation for each year. The relevant columns were renamed and organized into three variables: `Area` (country/region), `Year` (year), and `Population` (population).

**Data Quality Assessment:** In the original World Bank dataset, there was one missing record for each year from 2018 to 2025. However, after filtering the dataset to include only the 15 study countries, no missing values were found in the population data. The final population dataset contains 120 records, with no duplicate entries and no non-positive population values.

The cleaned population data will be merged with datasets from Ember and IRENA to facilitate cross-country comparisons and to calculate per capita solar PV installed capacity.

**Merge All Four Datasets**

In [98]:
##Check Cleaned Datasets

print("Solar Exports:", solar_annual.shape)
print("Cleantech Exports:", cleantech_annual.shape)
print("IRENA:", irena_annual.shape)
print("Population:", population_annual.shape)

print(solar_annual.columns)
print(cleantech_annual.columns)
print(irena_annual.columns)
print(population_annual.columns)


Solar Exports: (120, 4)
Cleantech Exports: (120, 4)
IRENA: (120, 4)
Population: (120, 3)
Index(['Area', 'Year', 'China_panel_exports_USD', 'China_panel_exports_MW'], dtype='str')
Index(['Area', 'Year', 'CleanTech_solarPV_USD', 'CleanTech_alltech_USD'], dtype='str')
Index(['Area', 'Year', 'PV_installed_capacity_MW', 'PV_generation_GWh'], dtype='str')
Index(['Area', 'Year', 'Population'], dtype='str')


In [99]:
##Merge Solar Exports and Cleantech Exports

solar_cleantech = pd.merge(
    solar_annual,
    cleantech_annual,
    on=["Area", "Year"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print(solar_cleantech["_merge"].value_counts())
print(solar_cleantech.shape)

##Remove merge indicator
solar_cleantech = solar_cleantech.drop(columns=["_merge"])


_merge
both          120
left_only       0
right_only      0
Name: count, dtype: int64
(120, 7)


In [100]:
##Merge with IRENA
solar_irena = pd.merge(
    solar_cleantech,
    irena_annual,
    on=["Area", "Year"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print(solar_irena["_merge"].value_counts())
print(solar_irena.shape)

##Remove merge indicator
solar_irena = solar_irena.drop(columns=["_merge"])


_merge
both          120
left_only       0
right_only      0
Name: count, dtype: int64
(120, 9)


In [101]:
##Merge with Population
solar_merged = pd.merge(
    solar_irena,
    population_annual,
    on=["Area", "Year"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print(solar_merged["_merge"].value_counts())
print(solar_merged.shape)

##Remove merge indicator
solar_merged = solar_merged.drop(columns=["_merge"])

solar_merged.head()


_merge
both          120
left_only       0
right_only      0
Name: count, dtype: int64
(120, 10)


,Area,Year,China_panel_exports_USD,China_panel_exports_MW,CleanTech_solarPV_USD,CleanTech_alltech_USD,PV_installed_capacity_MW,PV_generation_GWh,Population
0,Argentina,2018,171072125.0,665.24,171072125,636153927,196.11,123.48,44654882.0
1,Argentina,2019,134984757.0,496.16,134984757,586387276,445.85,860.07,44973465.0
2,Argentina,2020,12443409.0,52.89,12443409,255363464,767.87,1464.14,45191965.0
3,Argentina,2021,33380468.0,126.72,33380468,333211051,1076.92,2402.05,45312281.0
4,Argentina,2022,40100211.0,149.13,40233974,417618123,1112.67,3137.39,45407904.0


In [102]:
##Add Study Scope
final_data = solar_merged.copy()

##Assign South America as the default
final_data["Study_scope"] = "South America"

##Identify North American countries
north_america = [
    "Canada",
    "Mexico",
    "United States"
]

##Assign North America
final_data.loc[
    final_data["Area"].isin(north_america),
    "Study_scope"
] = "North America"


print(
    final_data[["Area", "Study_scope"]]
    .drop_duplicates()
    .sort_values("Area")
)


              Area    Study_scope
0        Argentina  South America
8          Bolivia  South America
16          Brazil  South America
24          Canada  North America
32           Chile  South America
40        Colombia  South America
48         Ecuador  South America
56          Guyana  South America
64          Mexico  North America
72        Paraguay  South America
80            Peru  South America
88        Suriname  South America
96   United States  North America
104        Uruguay  South America
112      Venezuela  South America


In [103]:
##Calculate Annual PV Capacity Additions

##Sort by country and year
final_data = final_data.sort_values(
    ["Area", "Year"]
).copy()

##Calculate annual capacity additions
final_data["PV_capacity_additions_MW"] = (
    final_data.groupby("Area")["PV_installed_capacity_MW"]
    .diff()
    .round(2)
)

final_data[[
    "Area",
    "Year",
    "PV_installed_capacity_MW",
    "PV_capacity_additions_MW"
]].head(10)


,Area,Year,PV_installed_capacity_MW,PV_capacity_additions_MW
0,Argentina,2018,196.11,NaN
1,Argentina,2019,445.85,249.74
2,Argentina,2020,767.87,322.02
3,Argentina,2021,1076.92,309.05
4,Argentina,2022,1112.67,35.75
5,Argentina,2023,1405.30,292.63
6,Argentina,2024,1740.60,335.30
7,Argentina,2025,2591.75,851.15
8,Bolivia,2018,70.50,NaN
9,Bolivia,2019,71.00,0.50


In [104]:
##Calculate Per-Capita PV Capacity

final_data["PV_installed_W_per_person"] = (
    final_data["PV_installed_capacity_MW"] * 1000000
    / final_data["Population"]
).round(6)


final_data[[
    "Area",
    "Year",
    "Population",
    "PV_installed_capacity_MW",
    "PV_installed_W_per_person"
]].head()


,Area,Year,Population,PV_installed_capacity_MW,PV_installed_W_per_person
0,Argentina,2018,44654882.0,196.11,4.391681
1,Argentina,2019,44973465.0,445.85,9.913624
2,Argentina,2020,45191965.0,767.87,16.991295
3,Argentina,2021,45312281.0,1076.92,23.766625
4,Argentina,2022,45407904.0,1112.67,24.503884


In [105]:
##Organize Final Dataset

##Select columns in the final order
final_data = final_data.loc[:, [
    "Area",
    "Study_scope",
    "Year",
    "China_panel_exports_USD",
    "China_panel_exports_MW",
    "CleanTech_solarPV_USD",
    "CleanTech_alltech_USD",
    "PV_installed_capacity_MW",
    "PV_generation_GWh",
    "Population",
    "PV_capacity_additions_MW",
    "PV_installed_W_per_person"
]].copy()

##Sort data
final_data = final_data.sort_values(
    ["Study_scope", "Area", "Year"]
).reset_index(drop=True)

print(final_data.shape)

final_data.head()


(120, 12)


,Area,Study_scope,Year,China_panel_exports_USD,China_panel_exports_MW,CleanTech_solarPV_USD,CleanTech_alltech_USD,PV_installed_capacity_MW,PV_generation_GWh,Population,PV_capacity_additions_MW,PV_installed_W_per_person
0,Canada,North America,2018,48086354.0,181.13,48121174,324210505,2631.0,3491.0,37072620.0,NaN,70.968817
1,Canada,North America,2019,62046149.0,235.58,62046216,501304576,2828.0,3752.0,37618495.0,197.0,75.175788
2,Canada,North America,2020,72110171.0,310.85,72110171,523513306,3068.0,4072.0,38028638.0,240.0,80.676042
3,Canada,North America,2021,59623578.0,222.12,59623578,857777248,3854.0,5115.0,38239864.0,786.0,100.784877
4,Canada,North America,2022,24636922.0,89.66,77003130,1107503288,4322.0,5737.0,38950132.0,468.0,110.962397


In [106]:
##Final Data Quality Audit

print("Shape:", final_data.shape)

print("\nMissing Values:")
print(final_data.isna().sum())

##Check duplicate country-year observations
print("\nDuplicate Country-Year Records:")
print(final_data.duplicated(
    subset=["Area", "Year"]
).sum())

##Check study coverage
print("\nNumber of Countries:", final_data["Area"].nunique())
print("Number of Years:", final_data["Year"].nunique())

##Check number of observations per year
print("\nObservations per Year:")
print(final_data["Year"].value_counts().sort_index())


Shape: (120, 12)

Missing Values:
Area                          0
Study_scope                   0
Year                          0
China_panel_exports_USD       0
China_panel_exports_MW        0
CleanTech_solarPV_USD         0
CleanTech_alltech_USD         0
PV_installed_capacity_MW      0
PV_generation_GWh            15
Population                    0
PV_capacity_additions_MW     15
PV_installed_W_per_person     0
dtype: int64

Duplicate Country-Year Records:
0

Number of Countries: 15
Number of Years: 8

Observations per Year:
Year
2018    15
2019    15
2020    15
2021    15
2022    15
2023    15
2024    15
2025    15
Name: count, dtype: int64


In [107]:
##Check Implausible Values

##Check negative solar exports
print(final_data.loc[
    final_data["China_panel_exports_MW"] < 0
])

##Check negative installed capacity
print(final_data.loc[
    final_data["PV_installed_capacity_MW"] < 0
])

##Check zero or negative population
print(final_data.loc[
    final_data["Population"] <= 0
])

##Inspect negative capacity additions
print(final_data.loc[
    final_data["PV_capacity_additions_MW"] < 0,
    ["Area", "Year", "PV_capacity_additions_MW"]
])


Empty DataFrame
Columns: [Area, Study_scope, Year, China_panel_exports_USD, China_panel_exports_MW, CleanTech_solarPV_USD, CleanTech_alltech_USD, PV_installed_capacity_MW, PV_generation_GWh, Population, PV_capacity_additions_MW, PV_installed_W_per_person]
Index: []
Empty DataFrame
Columns: [Area, Study_scope, Year, China_panel_exports_USD, China_panel_exports_MW, CleanTech_solarPV_USD, CleanTech_alltech_USD, PV_installed_capacity_MW, PV_generation_GWh, Population, PV_capacity_additions_MW, PV_installed_W_per_person]
Index: []
Empty DataFrame
Columns: [Area, Study_scope, Year, China_panel_exports_USD, China_panel_exports_MW, CleanTech_solarPV_USD, CleanTech_alltech_USD, PV_installed_capacity_MW, PV_generation_GWh, Population, PV_capacity_additions_MW, PV_installed_W_per_person]
Index: []
      Area  Year  PV_capacity_additions_MW
75  Guyana  2021                     -0.17
91    Peru  2021                     -2.48
92    Peru  2022                     -0.04


In [108]:
##Import Previous Excel Results

excel_final = pd.read_csv(
    processed_dir / "CLEANED_Study_area_data_2018_2025.csv"
)

print("Previous Excel:", excel_final.shape)
print("Python:", final_data.shape)


Previous Excel: (120, 12)
Python: (120, 12)


In [109]:
##Save Processed Dataset

##Save to Processed folder
final_data.to_csv(
    processed_dir / "solar_country_year_python_2018_2025.csv",
    index=False
)
